# 6.1 · 授权：MCP 中的 OAuth 2.1

**Authorization: OAuth 2.1 in MCP**

> **学完本课你能：** 带领 MCP 客户端走完完整的 OAuth 2.1 流程——401 质询、元数据发现、客户端注册、PKCE、与资源绑定的令牌、iss 校验以及权限范围递进授权——并保护一个校验受众与权限范围的服务器。

官方阅读材料：[Understanding authorization in MCP](https://modelcontextprotocol.io/docs/2026-07-28/tutorials/security/authorization) · [Spec: Authorization](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization/index) · [Spec: Authorization server discovery](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization/authorization-server-discovery) · [Spec: Client registration](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization/client-registration)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 手动实现 PKCE
PKCE 不过是一个随机密钥及其 SHA-256 哈希。客户端在浏览器重定向（可能泄露）中发送 <strong>challenge</strong>，而把 <strong>verifier</strong> 一直保留到后端信道的令牌请求（Token Request）时才使用。

In [ ]:
import base64
import hashlib
import json
import logging
import secrets

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpx2").setLevel(logging.WARNING)
logging.getLogger("mcp").setLevel(logging.ERROR)


def b64url(raw: bytes) -> str:
    return base64.urlsafe_b64encode(raw).rstrip(b"=").decode()


def make_pkce() -> tuple[str, str]:
    verifier = b64url(secrets.token_bytes(48))              # 64 chars, within the 43..128 allowed
    challenge = b64url(hashlib.sha256(verifier.encode()).digest())
    return verifier, challenge


verifier, challenge = make_pkce()
print("code_verifier :", verifier, f"({len(verifier)} chars, stays on the client)")
print("code_challenge:", challenge, "(goes in the authorization URL)")
print("method        : S256")
# the AS later recomputes the hash from the verifier the client presents:
print("verifies?     :", b64url(hashlib.sha256(verifier.encode()).digest()) == challenge)

### 2. 一个玩具授权服务器
真实部署会使用 Keycloak、Entra ID、Auth0……这里由一个约 60 行的 Starlette 应用扮演授权服务器（Authorization Server），让你看清每个字段。它发布 RFC 8414 元数据（带有 `code_challenge_methods_supported` 和 `authorization_response_iss_parameter_supported`），在 `/authorize` 自动批准用户 “alice”（代替登录 + 同意页面），在 `/token` 强制执行重定向 URI 精确匹配和 PKCE，并把请求中的 `resource` 写进令牌的 `aud`。令牌是 HMAC 签名的 JWT——用于教学没问题，<strong>生产环境中绝不要自己手写这些</strong>。

In [ ]:
import hmac
import socket
import threading
import time
from urllib.parse import parse_qs, urlencode, urlparse

import uvicorn
from starlette.applications import Starlette
from starlette.responses import JSONResponse, RedirectResponse
from starlette.routing import Route


def free_port() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def serve(app, port):
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.02)
    return server


AS_PORT, MCP_PORT = free_port(), free_port()
ISSUER = f"http://127.0.0.1:{AS_PORT}"
RESOURCE = f"http://127.0.0.1:{MCP_PORT}/mcp"          # the MCP server's canonical URI
SIGNING_KEY = secrets.token_bytes(32)
REDIRECT_URI = "http://127.0.0.1:33418/callback"


def mint_jwt(claims: dict) -> str:
    head = b64url(json.dumps({"alg": "HS256", "typ": "JWT"}).encode())
    body = b64url(json.dumps(claims).encode())
    sig = b64url(hmac.new(SIGNING_KEY, f"{head}.{body}".encode(), hashlib.sha256).digest())
    return f"{head}.{body}.{sig}"


def read_jwt(token: str) -> dict | None:
    """Return the claims if the signature is valid, else None."""
    try:
        head, body, sig = token.split(".")
        good = b64url(hmac.new(SIGNING_KEY, f"{head}.{body}".encode(), hashlib.sha256).digest())
        if not hmac.compare_digest(sig, good):
            return None
        return json.loads(base64.urlsafe_b64decode(body + "=" * (-len(body) % 4)))
    except ValueError:
        return None


CLIENTS = {"notes-cli": {"redirect_uris": [REDIRECT_URI]}}   # pre-registered clients
CODES: dict[str, dict] = {}
REFRESH: dict[str, dict] = {}


async def as_metadata(request):
    return JSONResponse({
        "issuer": ISSUER,
        "authorization_endpoint": f"{ISSUER}/authorize",
        "token_endpoint": f"{ISSUER}/token",
        "response_types_supported": ["code"],
        "grant_types_supported": ["authorization_code", "refresh_token"],
        "code_challenge_methods_supported": ["S256"],
        "token_endpoint_auth_methods_supported": ["none"],
        "scopes_supported": ["notes:read", "notes:write"],
        "authorization_response_iss_parameter_supported": True,
    })


async def authorize(request):
    q = request.query_params
    client = CLIENTS.get(q.get("client_id"))
    if client is None or q.get("redirect_uri") not in client["redirect_uris"]:   # exact match only
        return JSONResponse({"error": "invalid_request", "error_description": "unknown client or redirect_uri"}, 400)
    if q.get("code_challenge_method") != "S256" or not q.get("code_challenge"):
        return JSONResponse({"error": "invalid_request", "error_description": "PKCE S256 required"}, 400)
    # ... a real AS shows a login + consent page here; we auto-approve user "alice" ...
    code = secrets.token_urlsafe(16)
    CODES[code] = {"client_id": q["client_id"], "redirect_uri": q["redirect_uri"], "challenge": q["code_challenge"],
                   "scope": q.get("scope", ""), "resource": q.get("resource"), "sub": "alice"}
    back = {"code": code, "state": q.get("state", ""), "iss": ISSUER}             # RFC 9207 iss
    return RedirectResponse(f"{q['redirect_uri']}?{urlencode(back)}", status_code=302)


def issue_tokens(sub, client_id, scope, resource, ttl=300):
    now = int(time.time())
    access = mint_jwt({"iss": ISSUER, "sub": sub, "aud": resource, "client_id": client_id,
                       "scope": scope, "iat": now, "exp": now + ttl})
    refresh = secrets.token_urlsafe(24)
    REFRESH[refresh] = {"sub": sub, "client_id": client_id, "scope": scope, "resource": resource}
    return {"access_token": access, "token_type": "Bearer", "expires_in": ttl, "refresh_token": refresh, "scope": scope}


async def token(request):
    form = await request.form()
    if form.get("grant_type") == "authorization_code":
        rec = CODES.pop(form.get("code"), None)                                   # codes are single-use
        if rec is None or rec["client_id"] != form.get("client_id") or rec["redirect_uri"] != form.get("redirect_uri"):
            return JSONResponse({"error": "invalid_grant"}, 400)
        if b64url(hashlib.sha256(form.get("code_verifier", "").encode()).digest()) != rec["challenge"]:
            return JSONResponse({"error": "invalid_grant", "error_description": "PKCE verification failed"}, 400)
        if form.get("resource") != rec["resource"]:
            return JSONResponse({"error": "invalid_target"}, 400)
        return JSONResponse(issue_tokens(rec["sub"], rec["client_id"], rec["scope"], rec["resource"]))
    if form.get("grant_type") == "refresh_token":
        rec = REFRESH.pop(form.get("refresh_token"), None)                         # rotation: old one dies
        if rec is None:
            return JSONResponse({"error": "invalid_grant", "error_description": "unknown or reused refresh token"}, 400)
        return JSONResponse(issue_tokens(rec["sub"], rec["client_id"], rec["scope"], rec["resource"]))
    return JSONResponse({"error": "unsupported_grant_type"}, 400)


as_app = Starlette(routes=[Route("/.well-known/oauth-authorization-server", as_metadata),
                           Route("/authorize", authorize),
                           Route("/token", token, methods=["POST"])])
as_server = serve(as_app, AS_PORT)
print("authorization server:", ISSUER)

### 3. 用 SDK 搭建受保护的 MCP 服务器
SDK 负责资源服务器（Resource Server）的底层管线：给定 `token_verifier=` 和 `auth=AuthSettings(...)`，它会发布 PRM，用 `401` + `resource_metadata` 回应未认证的请求，并且——在 `validate_token_resource=True` 时——拒绝 `AccessToken.resource` 不等于 `resource_server_url` 的令牌。`required_scopes` 作用于<em>每一个</em>请求，因此按工具区分的权限范围（Scope）需要你自己检查：一个小小的 ASGI 包装层读取 2026-07-28 的 `Mcp-Method` / `Mcp-Name` 请求头，除非令牌拥有 `notes:write`，否则对 `delete_note` 返回 `403 insufficient_scope`。

In [ ]:
from mcp.server import MCPServer
from mcp.server.auth.provider import AccessToken
from mcp.server.auth.settings import AuthSettings


class HmacTokenVerifier:
    """Signature + issuer + expiry. The SDK then compares `resource` (the aud) with resource_server_url."""

    async def verify_token(self, token: str) -> AccessToken | None:
        claims = read_jwt(token)
        if not claims or claims.get("iss") != ISSUER or claims.get("exp", 0) < time.time():
            return None
        return AccessToken(token=token, client_id=claims["client_id"], scopes=claims["scope"].split(),
                           expires_at=claims["exp"], resource=claims["aud"], subject=claims["sub"])


notes_server = MCPServer(
    "notes",
    token_verifier=HmacTokenVerifier(),
    auth=AuthSettings(issuer_url=ISSUER, resource_server_url=RESOURCE,
                      required_scopes=["notes:read"], validate_token_resource=True),
)
NOTES = {"n1": "buy milk", "n2": "renew passport"}


@notes_server.tool()
def list_notes() -> list[str]:
    """List the ids of the user's notes."""
    return sorted(NOTES)


@notes_server.tool()
def delete_note(note_id: str) -> str:
    """Delete one note (needs notes:write)."""
    NOTES.pop(note_id, None)
    return f"deleted {note_id}"


PRM_URL = f"http://127.0.0.1:{MCP_PORT}/.well-known/oauth-protected-resource/mcp"
TOOL_SCOPES = {"delete_note": "notes:write"}
inner_app = notes_server.streamable_http_app()


async def scope_gate(scope, receive, send):
    """Per-tool scope check that emits a spec-shaped 403 challenge."""
    if scope["type"] == "http":
        headers = {k.decode().lower(): v.decode() for k, v in scope["headers"]}
        needed = TOOL_SCOPES.get(headers.get("mcp-name")) if headers.get("mcp-method") == "tools/call" else None
        auth = headers.get("authorization", "")
        if needed and auth.lower().startswith("bearer "):
            claims = read_jwt(auth[7:]) or {}
            if needed not in claims.get("scope", "").split():
                challenge = (f'Bearer error="insufficient_scope", scope="{needed}", '
                             f'resource_metadata="{PRM_URL}", error_description="{needed} required"')
                await send({"type": "http.response.start", "status": 403,
                            "headers": [(b"content-type", b"application/json"),
                                        (b"www-authenticate", challenge.encode())]})
                await send({"type": "http.response.body", "body": b'{"error": "insufficient_scope"}'})
                return
    await inner_app(scope, receive, send)        # lifespan and everything else pass straight through


mcp_server = serve(scope_gate, MCP_PORT)
print("protected MCP server:", RESOURCE)

### 4. 第 1 步——401 质询
像一个一无所知的客户端那样调用服务器。有意思的部分是 `WWW-Authenticate` 头。

In [ ]:
import re

import httpx

http = httpx.AsyncClient(trust_env=False)
MCP_HEADERS = {"Accept": "application/json, text/event-stream", "MCP-Protocol-Version": "2026-07-28",
               "Mcp-Method": "tools/list"}
LIST_TOOLS = {"jsonrpc": "2.0", "id": 1, "method": "tools/list",
              "params": {"_meta": {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
                                   "io.modelcontextprotocol/clientCapabilities": {}}}}


def parse_www_authenticate(value: str) -> dict:
    return dict(re.findall(r'(\w+)="([^"]*)"', value or ""))


r = await http.post(RESOURCE, json=LIST_TOOLS, headers=MCP_HEADERS)
print("HTTP", r.status_code)
print("WWW-Authenticate:", r.headers["www-authenticate"])
challenge = parse_www_authenticate(r.headers["www-authenticate"])
print(json.dumps(challenge, indent=2))

注意，SDK 在 401 上总是加上 `error="invalid_token"`，却从不带 `scope=`。规范只说服务器<strong>应当（SHOULD）</strong>包含 `scope`；没有它时，客户端会回退到 PRM 的 `scopes_supported`。

### 5. 第 2–3 步——先取受保护资源元数据，再取授权服务器元数据
顺着指针走，然后按规范规定的顺序构造授权服务器的 well-known URL。客户端<strong>必须</strong>做两项检查：文档中的 `issuer` 等于我们用来构造 URL 的颁发者（Issuer），以及声明了对 PKCE 的支持。

In [ ]:
prm = (await http.get(challenge["resource_metadata"])).json()
print("PRM:", json.dumps(prm, indent=2))


def as_metadata_urls(issuer: str) -> list[str]:
    u = urlparse(issuer)
    base, path = f"{u.scheme}://{u.netloc}", u.path.rstrip("/")
    if path:
        return [f"{base}/.well-known/oauth-authorization-server{path}",
                f"{base}/.well-known/openid-configuration{path}",
                f"{base}{path}/.well-known/openid-configuration"]
    return [f"{base}/.well-known/oauth-authorization-server", f"{base}/.well-known/openid-configuration"]


print("\ndiscovery order for a tenant issuer:", *as_metadata_urls("https://auth.example.com/tenant1"), sep="\n  ")

as_issuer = prm["authorization_servers"][0]
for url in as_metadata_urls(as_issuer):
    resp = await http.get(url)
    if resp.status_code == 200:
        asm = resp.json()
        print("\nfound AS metadata at", url)
        break

assert asm["issuer"] == as_issuer, "issuer mismatch -> MUST NOT use this metadata"
assert "S256" in asm.get("code_challenge_methods_supported", []), "no PKCE support -> MUST refuse"
print("issuer matches, S256 supported, iss-in-response advertised:",
      asm.get("authorization_response_iss_parameter_supported"))

### 6. 第 4 步——选择注册机制
把客户端注册页面中的优先级顺序写成一个函数。我们的授权服务器既没有 `registration_endpoint`，也不支持 CIMD，但我们已以 `notes-cli` 预注册（Pre-registration）——并绑定到这个颁发者。

In [ ]:
PRE_REGISTERED = {ISSUER: "notes-cli"}           # keyed by issuer: credentials are bound to their AS


def choose_registration(asm: dict, cimd_url: str | None) -> str:
    if asm["issuer"] in PRE_REGISTERED:
        return f"pre-registered client_id={PRE_REGISTERED[asm['issuer']]}"
    if cimd_url and asm.get("client_id_metadata_document_supported"):
        return f"CIMD: client_id={cimd_url}"
    if asm.get("registration_endpoint"):
        return "DCR (deprecated): POST registration_endpoint with application_type='native'"
    return "ask the user to enter client information"


print(choose_registration(asm, "https://app.example.com/oauth/client.json"))
print(choose_registration({"issuer": "https://other.example", "client_id_metadata_document_supported": True},
                          "https://app.example.com/oauth/client.json"))
print(choose_registration({"issuer": "https://legacy.example", "registration_endpoint": "https://legacy.example/register"}, None))

### 7. 第 5–6 步——授权请求、回调、`state` 与 `iss`
我们用 `httpx`（不跟随重定向）扮演浏览器，并读取授权服务器发回的 `Location` 头。权限范围（Scope）选择：401 中没有 `scope`，所以我们采用 PRM 的 `scopes_supported`。在使用授权码之前，我们检查 `state` 并套用 RFC 9207 的那张表。

In [ ]:
def check_iss(as_meta: dict, recorded_issuer: str, iss: str | None) -> None:
    """The 2026-07-28 Authorization Response Validation table."""
    supported = as_meta.get("authorization_response_iss_parameter_supported") is True
    if iss is not None:
        if iss != recorded_issuer:                 # simple string comparison, no normalisation
            raise ValueError(f"iss mismatch: {iss!r} != {recorded_issuer!r} (possible mix-up attack)")
    elif supported:
        raise ValueError("iss missing although the AS advertises it")


verifier, challenge_s256 = make_pkce()
state = secrets.token_urlsafe(16)
pending = {"verifier": verifier, "state": state, "issuer": asm["issuer"]}      # recorded BEFORE redirecting
scope = challenge.get("scope") or " ".join(prm.get("scopes_supported", []))

auth_url = asm["authorization_endpoint"] + "?" + urlencode({
    "response_type": "code", "client_id": "notes-cli", "redirect_uri": REDIRECT_URI, "state": state,
    "code_challenge": challenge_s256, "code_challenge_method": "S256", "scope": scope, "resource": RESOURCE})
print("authorization URL:\n ", auth_url, "\n")

redirect = await http.get(auth_url)
callback = {k: v[0] for k, v in parse_qs(urlparse(redirect.headers["location"]).query).items()}
print("callback params:", json.dumps(callback, indent=2))

assert callback["state"] == pending["state"], "state mismatch -> discard"
check_iss(asm, pending["issuer"], callback.get("iss"))
print("state and iss OK")
for bad_iss in ["http://evil.example", ISSUER + "/", None]:
    try:
        check_iss(asm, pending["issuer"], bad_iss)
    except ValueError as exc:
        print("rejected:", exc)

末尾斜杠的情况是故意被拒绝的：规范禁止在比较之前做规范化。

### 8. 第 7 步——兑换授权码
令牌请求携带 `code_verifier` 和同一个 `resource`。解码访问令牌即可看到受众（Audience）绑定。

In [ ]:
token_endpoint = asm["token_endpoint"]
form = {"grant_type": "authorization_code", "code": callback["code"], "redirect_uri": REDIRECT_URI,
        "client_id": "notes-cli", "code_verifier": pending["verifier"], "resource": RESOURCE}
tokens = (await http.post(token_endpoint, data=form)).json()
print({k: (v[:25] + "…" if isinstance(v, str) and len(v) > 25 else v) for k, v in tokens.items()})
print("access token claims:", json.dumps(read_jwt(tokens["access_token"]), indent=2))

replay = await http.post(token_endpoint, data=form)          # codes are single-use
print("replaying the same code ->", replay.status_code, replay.json())

### 9. 第 8 步——带着令牌调用服务器
`Client(url)` 会构建自己的 HTTP 客户端；要添加请求头，就传入一个用 `streamable_http_client(url, http_client=...)` 构建的传输（Transport）（SDK 使用 `httpx2`）。<strong>每个</strong>请求上都带同一个令牌。

In [ ]:
import httpx2

from mcp import Client
from mcp.client.streamable_http import streamable_http_client

access_token = tokens["access_token"]
async with httpx2.AsyncClient(headers={"Authorization": f"Bearer {access_token}"}, trust_env=False) as hc:
    async with Client(streamable_http_client(RESOURCE, http_client=hc)) as client:
        print("tools:", [t.name for t in (await client.list_tools()).tools])
        result = await client.call_tool("list_notes", {})
        print("list_notes ->", result.structured_content)

# the same thing on the wire
r = await http.post(RESOURCE, json=LIST_TOOLS, headers={**MCP_HEADERS, "Authorization": f"Bearer {access_token}"})
print("\nraw HTTP", r.status_code, r.headers["content-type"])
print(json.dumps(r.json()["result"]["tools"][0], indent=2)[:400])

### 10. 搞破坏——服务器必须拒绝的令牌
下面每一个看起来都是合法令牌。只有受众（Audience）绑定正确、未被篡改、未过期、且来自<em>本</em>颁发者的那一个能用。

In [ ]:
now = int(time.time())
good = read_jwt(access_token)
candidates = {
    "correct token": access_token,
    "audience = another service": mint_jwt({**good, "aud": "https://payroll.example.com/mcp"}),
    "expired": mint_jwt({**good, "exp": now - 10}),
    "other issuer (same key!)": mint_jwt({**good, "iss": "https://other-tenant.example"}),
    "tampered scope": access_token.split(".")[0] + "." + b64url(json.dumps({**good, "scope": "notes:read notes:write"}).encode()) + "." + access_token.split(".")[2],
}
for label, tok in candidates.items():
    r = await http.post(RESOURCE, json=LIST_TOOLS, headers={**MCP_HEADERS, "Authorization": f"Bearer {tok}"})
    print(f"{label:28} -> {r.status_code}")

r = await http.post(RESOURCE + f"?access_token={access_token}", json=LIST_TOOLS, headers=MCP_HEADERS)
print(f"{'token in query string':28} -> {r.status_code}  (MUST NOT be used; the server ignores it)")

### 11. 手动完成权限范围质询与递进授权
我们的令牌只有 `notes:read`。调用 `delete_note` 会在运行时得到带所需权限范围（Scope）的 <strong>403</strong>。客户端以新旧权限范围的<strong>并集</strong>重新授权并重试——这就是递进授权（Step-up Authorization）。

In [ ]:
call = {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
        "params": {"name": "delete_note", "arguments": {"note_id": "n2"},
                   "_meta": LIST_TOOLS["params"]["_meta"]}}
call_headers = {**MCP_HEADERS, "Mcp-Method": "tools/call", "Mcp-Name": "delete_note"}

r = await http.post(RESOURCE, json=call, headers={**call_headers, "Authorization": f"Bearer {access_token}"})
print("HTTP", r.status_code, "\nWWW-Authenticate:", r.headers["www-authenticate"])
step = parse_www_authenticate(r.headers["www-authenticate"])

granted = set(tokens["scope"].split())
wanted = " ".join(sorted(granted | set(step["scope"].split())))      # union, so notes:read is not lost
print("re-authorizing with scope =", repr(wanted))


async def authorize_and_redeem(scope: str) -> dict:
    v, c = make_pkce()
    s = secrets.token_urlsafe(16)
    loc = (await http.get(asm["authorization_endpoint"] + "?" + urlencode({
        "response_type": "code", "client_id": "notes-cli", "redirect_uri": REDIRECT_URI, "state": s,
        "code_challenge": c, "code_challenge_method": "S256", "scope": scope, "resource": RESOURCE}))).headers["location"]
    cb = {k: x[0] for k, x in parse_qs(urlparse(loc).query).items()}
    assert cb["state"] == s
    check_iss(asm, asm["issuer"], cb.get("iss"))
    return (await http.post(asm["token_endpoint"], data={
        "grant_type": "authorization_code", "code": cb["code"], "redirect_uri": REDIRECT_URI,
        "client_id": "notes-cli", "code_verifier": v, "resource": RESOURCE})).json()


tokens = await authorize_and_redeem(wanted)
r = await http.post(RESOURCE, json=call, headers={**call_headers, "Authorization": f"Bearer {tokens['access_token']}"})
print("retry ->", r.status_code, r.json()["result"]["content"][0]["text"])

### 12. 刷新令牌轮换
访问令牌短期有效；刷新令牌（Refresh Token）无需浏览器即可换来新的访问令牌。我们的授权服务器会轮换刷新令牌（规范对公共客户端有此要求），因此被盗且已被使用过的刷新令牌就作废了。

In [ ]:
old_refresh = tokens["refresh_token"]
fresh = (await http.post(token_endpoint, data={"grant_type": "refresh_token", "refresh_token": old_refresh,
                                               "client_id": "notes-cli", "resource": RESOURCE})).json()
print("new access token aud/scope:", read_jwt(fresh["access_token"])["aud"], "|", fresh["scope"])
print("refresh token rotated:", fresh["refresh_token"] != old_refresh)
reuse = await http.post(token_endpoint, data={"grant_type": "refresh_token", "refresh_token": old_refresh,
                                              "client_id": "notes-cli", "resource": RESOURCE})
print("reusing the old refresh token ->", reuse.status_code, reuse.json())

### 13. 让 SDK 包办一切
`mcp.client.auth.OAuthClientProvider` 是一个 `httpx2.Auth`，它在遇到 401 时跑完整个流程——带回退的 PRM 发现、带颁发者校验的授权服务器元数据、注册、PKCE、`state` + `iss` 检查、带 `resource` 的令牌请求——并在遇到 `403 insufficient_scope` 时执行<strong>递进授权（Step-up Authorization）</strong>。你只需提供存储（这里是内存存储，预注册并绑定到我们的颁发者）以及两个浏览器钩子。一个事件钩子会打印每一次 HTTP 交互。

In [ ]:
from mcp.client.auth import AuthorizationCodeResult, OAuthClientProvider
from mcp.shared.auth import OAuthClientInformationFull, OAuthClientMetadata

NOTES["n2"] = "renew passport"


class MemoryStorage:
    def __init__(self):
        self.tokens = None
        self.client = OAuthClientInformationFull(client_id="notes-cli", redirect_uris=[REDIRECT_URI],
                                                 token_endpoint_auth_method="none", issuer=ISSUER)

    async def get_tokens(self): return self.tokens
    async def set_tokens(self, tokens): self.tokens = tokens
    async def get_client_info(self): return self.client
    async def set_client_info(self, info): self.client = info


browser = {}


async def redirect_handler(url: str) -> None:          # a desktop app would open the system browser
    print("    [browser] user approves at", url.split("?")[0])
    async with httpx.AsyncClient(trust_env=False) as b:
        browser["location"] = (await b.get(url)).headers["location"]


async def callback_handler() -> AuthorizationCodeResult:  # ...and listen on the redirect URI
    q = {k: v[0] for k, v in parse_qs(urlparse(browser["location"]).query).items()}
    return AuthorizationCodeResult(code=q["code"], state=q.get("state"), iss=q.get("iss"))


provider = OAuthClientProvider(
    server_url=RESOURCE,
    client_metadata=OAuthClientMetadata(client_name="notes-cli", redirect_uris=[REDIRECT_URI],
                                        grant_types=["authorization_code", "refresh_token"],
                                        token_endpoint_auth_method="none"),
    storage=MemoryStorage(), redirect_handler=redirect_handler, callback_handler=callback_handler)


async def log_exchange(response):
    req = response.request
    print(f"  {req.method:4} {req.url.path:48} -> {response.status_code}  {req.headers.get('mcp-method', '')}")


async with httpx2.AsyncClient(auth=provider, trust_env=False, event_hooks={"response": [log_exchange]}) as hc:
    async with Client(streamable_http_client(RESOURCE, http_client=hc)) as client:
        print("list_notes ->", (await client.call_tool("list_notes", {})).structured_content)
        print("granted scope:", provider.context.current_tokens.scope, "\n")
        r = await client.call_tool("delete_note", {"note_id": "n2"})
        print("delete_note ->", r.content[0].text)
        print("granted scope:", provider.context.current_tokens.scope)

看日志：一次 401、PRM、授权服务器元数据、浏览器、`/token`、重试；之后一次 403、带着权限范围<strong>并集</strong>再次打开浏览器、`/token`、重试。SDK 有两个值得了解的怪癖：它在服务器端（由 `required_scopes` 产生）的 403 不带 `scope=` 参数；而客户端 provider 本身并不会拒绝元数据缺少 `code_challenge_methods_supported` 的授权服务器——如果你的授权服务器可能这样，就自己检查。

In [ ]:
await http.aclose()
mcp_server.should_exit = True
as_server.should_exit = True
print("servers stopped")

### 动手练习
1. 让授权服务器“忘掉” PKCE：从 `as_metadata` 中删除 `code_challenge_methods_supported`，重新运行第 5 节，并编写拒绝继续的客户端检查。然后确认第 13 节中的 SDK provider <em>不会</em>拒绝——并为它加一道防护。
2. 修改玩具授权服务器，让它无论 `resource` 是什么，都在每个令牌中写入 `"aud": "https://api.example.com"`（即 Keycloak 教程中固定受众的捷径）。哪一节会失败？生产环境中的授权服务器又应该怎么做？
3. 让 `scope_gate` 为 `delete_note` 返回<strong>两个</strong>缺失的权限范围（`notes:write audit:write`），并确认 SDK 在一次递进授权中请求三者的并集，而不是分两次。
4. 在 PRM 的 `authorization_servers` 中再加一项。客户端在流程的哪一步做出选择？为什么它为第一个授权服务器保存的 `client_id` 绝不能发给第二个？